<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Use Case 8 - Classification Of Trips
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<h3>Overview:</h3>
<p>The following notebook shows how to use XGBoost within a Teradata database through the TD_XGBoost and TD_XGBoostPredict functions. XGBoost is used to classify shopping trips for marketing purposes. The notebook follows the following steps:</p>

<li>Import the required libraries</li>
<li>Define the columns</li>
<li>Function to connect to the database</li>
<li>Function to pre process the data</li>
<li>Function to train the XGBoost model on the data</li>
<li>Function to serve the model</li>
<li>Main function that runs all the above functions and times them</li>

<h3><b>Import the required libraries</b></h3>

In [1]:
#Import the required libraries
import argparse
import datetime
import os
import timeit
import warnings
from pathlib import Path


In [2]:
# teradata lib
import teradataml
from teradataml import *
configure.val_install_location = 'val'

In [3]:
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

# for environment definitions
import settings


<h3><b>Define columns</b></h3>

In [4]:
# const
department_columns = [
    "FINANCIAL SERVICES", "SHOES", "PERSONAL CARE", "PAINT AND ACCESSORIES", "DSD GROCERY", "MEAT - FRESH & FROZEN",
    "DAIRY", "PETS AND SUPPLIES", "HOUSEHOLD CHEMICALS/SUPP", "IMPULSE MERCHANDISE", "PRODUCE",
    "CANDY, TOBACCO, COOKIES", "GROCERY DRY GOODS", "BOYS WEAR", "FABRICS AND CRAFTS", "JEWELRY AND SUNGLASSES",
    "MENS WEAR", "ACCESSORIES", "HOME MANAGEMENT", "FROZEN FOODS", "SERVICE DELI", "INFANT CONSUMABLE HARDLINES",
    "PRE PACKED DELI", "COOK AND DINE", "PHARMACY OTC", "LADIESWEAR", "COMM BREAD", "BAKERY", "HOUSEHOLD PAPER GOODS",
    "CELEBRATION", "HARDWARE", "BEAUTY", "AUTOMOTIVE", "BOOKS AND MAGAZINES", "SEAFOOD", "OFFICE SUPPLIES",
    "LAWN AND GARDEN", "SHEER HOSIERY", "WIRELESS", "BEDDING", "BATH AND SHOWER", "HORTICULTURE AND ACCESS",
    "HOME DECOR", "TOYS", "INFANT APPAREL", "LADIES SOCKS", "PLUS AND MATERNITY", "ELECTRONICS",
    "GIRLS WEAR, 4-6X  AND 7-14", "BRAS & SHAPEWEAR", "LIQUOR,WINE,BEER", "SLEEPWEAR/FOUNDATIONS",
    "CAMERAS AND SUPPLIES", "SPORTING GOODS", "PLAYERS AND ELECTRONICS", "PHARMACY RX", "MENSWEAR", "OPTICAL - FRAMES",
    "SWIMWEAR/OUTERWEAR", "OTHER DEPARTMENTS", "MEDIA AND GAMING", "FURNITURE", "OPTICAL - LENSES", "SEASONAL",
    "LARGE HOUSEHOLD GOODS", "1-HR PHOTO", "CONCEPT STORES", "HEALTH AND BEAUTY AIDS"
]

weekday_columns = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

featureColumns = ['scan_count', 'scan_count_abs'] + weekday_columns + department_columns

label_column = 'trip_type'

#View the columns
print("Feature Columns:")
for col in featureColumns:
    print(f"- {col}")


Feature Columns:
- scan_count
- scan_count_abs
- Monday
- Tuesday
- Wednesday
- Thursday
- Friday
- Saturday
- Sunday
- FINANCIAL SERVICES
- SHOES
- PERSONAL CARE
- PAINT AND ACCESSORIES
- DSD GROCERY
- MEAT - FRESH & FROZEN
- DAIRY
- PETS AND SUPPLIES
- HOUSEHOLD CHEMICALS/SUPP
- IMPULSE MERCHANDISE
- PRODUCE
- CANDY, TOBACCO, COOKIES
- GROCERY DRY GOODS
- BOYS WEAR
- FABRICS AND CRAFTS
- JEWELRY AND SUNGLASSES
- MENS WEAR
- ACCESSORIES
- HOME MANAGEMENT
- FROZEN FOODS
- SERVICE DELI
- INFANT CONSUMABLE HARDLINES
- PRE PACKED DELI
- COOK AND DINE
- PHARMACY OTC
- LADIESWEAR
- COMM BREAD
- BAKERY
- HOUSEHOLD PAPER GOODS
- CELEBRATION
- HARDWARE
- BEAUTY
- AUTOMOTIVE
- BOOKS AND MAGAZINES
- SEAFOOD
- OFFICE SUPPLIES
- LAWN AND GARDEN
- SHEER HOSIERY
- WIRELESS
- BEDDING
- BATH AND SHOWER
- HORTICULTURE AND ACCESS
- HOME DECOR
- TOYS
- INFANT APPAREL
- LADIES SOCKS
- PLUS AND MATERNITY
- ELECTRONICS
- GIRLS WEAR, 4-6X  AND 7-14
- BRAS & SHAPEWEAR
- LIQUOR,WINE,BEER
- SLEEPWEAR/FOUNDATION

<h3><b>Define a function to connect to the database</b></h3>
<p>Before running this code block create a file called "settings" that contains the following fields with your own credentials:</p>
<p>system_dsn=hostip or name</p>
<p>system_user=username</p>
<p>system_pw=password</p>

In [5]:
def connect_to_db():
    # Connect to the database
    create_context(host=settings.system_dsn, username=settings.system_user, password=settings.system_pw, logmech='LDAP')

<h3><b>Define a function to pre process the data</b></h3>
<p>There are 3 tables in the database initially:</p>
<li>Orders</li>
<li>Product</li>
<li>LineItem</li>

<p>The final table used in the XGBoost model is called: uc08_preprocessed</p>
<p>uc08_preprocessed contains all columns from the original 3 tables as well as some processed features like scan_count</p>

In [6]:
def pre_process():
    #print('pre_process called...')

    # drop the table first
    for t in [
        'uc08_order_lineitem',
        'uc08_order_lineitem_product',
        'uc08_order_lineitem_product_agg',
        'uc08_order_lineitem_product_pivot_dept',
        'uc08_order_lineitem_product_scan_count',
        'uc08_order_lineitem_product_wd',
        'uc08_order_lineitem_product_agg_wd2',
        'uc08_order_lineitem_product_agg_wd_dept',
        'uc08_preprocessed']:
        try:
            execute_sql(f"DROP TABLE TPCXAI.{t}")
        except Exception as e:
            pass

    # Query to create the order_lineitem table by joining Orders and LineItem tables
    qry_order_lineitem = """
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem AS (
    SELECT o.*,
        li.*
        FROM TPCXAI.Orders o
        JOIN TPCXAI.LineItem li ON o.o_order_id =  li.li_order_id
    ) WITH DATA
    PRIMARY INDEX(o_order_id);
    """
    execute_sql(qry_order_lineitem)

    # Query to create the order_lineitem_product table by joining order_lineitem and Product tables
    qry_order_lineitem_product_123 = """
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product AS (
    SELECT oli.o_order_id,
        oli.order_date,
        oli.quantity,
        oli.trip_type,
        p.department
        FROM TPCXAI.uc08_order_lineitem oli
        JOIN TPCXAI.Product p ON oli.li_product_id = p.p_product_id
    ) WITH DATA
    PRIMARY INDEX(o_order_id);
    """
    execute_sql(qry_order_lineitem_product_123)

    # Query to create the order_lineitem_product_agg table by aggregating the order_lineitem_product table
    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_agg AS (
    SELECT o_order_id,
        SUM(quantity) AS scan_count,
        ABS(SUM(quantity)) AS scan_count_abs,
        MIN(TD_DAY_OF_WEEK(CAST (order_date AS DATE))) AS dayofweek,
        MIN(trip_type) AS trip_type
        FROM TPCXAI.uc08_order_lineitem_product
        GROUP BY o_order_id
    ) WITH DATA;'''
    execute_sql(qry)

    # -- Pivot department
    # -- CALL P_DROP_OBJ('uc08_order_lineitem_product_pivot_dept', 'T');

    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_pivot_dept AS (
        SELECT *
            FROM TPCXAI.uc08_order_lineitem_product PIVOT (
            SUM(quantity) AS sc
            FOR department IN (
            'FINANCIAL SERVICES' AS FINANCIAL_SERVICES,
            'SHOES' AS SHOES,
            'PERSONAL CARE' AS PERSONAL_CARE,
            'PAINT AND ACCESSORIES' AS PAINT_AND_ACCESSORIES,
            'DSD GROCERY' AS DSD_GROCERY,
            'MEAT - FRESH & FROZEN' AS MEAT___FRESH__FROZEN,
            'DAIRY' AS DAIRY,
            'PETS AND SUPPLIES' AS PETS_AND_SUPPLIES,
            'HOUSEHOLD CHEMICALS/SUPP' AS HOUSEHOLD_CHEMICALS_SUPP,
            'IMPULSE MERCHANDISE' AS IMPULSE_MERCHANDISE,
            'PRODUCE' AS PRODUCE,
            'CANDY, TOBACCO, COOKIES' AS CANDY__TOBACCO__COOKIES,
            'GROCERY DRY GOODS' AS GROCERY_DRY_GOODS,
            'BOYS WEAR' AS BOYS_WEAR,
            'FABRICS AND CRAFTS' AS FABRICS_AND_CRAFTS,
            'JEWELRY AND SUNGLASSES' AS JEWELRY_AND_SUNGLASSES,
            'MENS WEAR' AS MENS_WEAR,
            'ACCESSORIES' AS ACCESSORIES,
            'HOME MANAGEMENT' AS HOME_MANAGEMENT,
            'FROZEN FOODS' AS FROZEN_FOODS,
            'SERVICE DELI' AS SERVICE_DELI,
            'INFANT CONSUMABLE HARDLINES' AS INFANT_CONSUMABLE_HARDLINES,
            'PRE PACKED DELI' AS PRE_PACKED_DELI,
            'COOK AND DINE' AS COOK_AND_DINE,
            'PHARMACY OTC' AS PHARMACY_OTC,
            'LADIESWEAR' AS LADIESWEAR,
            'COMM BREAD' AS COMM_BREAD,
            'BAKERY' AS BAKERY,
            'HOUSEHOLD PAPER GOODS' AS HOUSEHOLD_PAPER_GOODS,
            'CELEBRATION' AS CELEBRATION,
            'HARDWARE' AS HARDWARE,
            'BEAUTY' AS BEAUTY,
            'AUTOMOTIVE' AS AUTOMOTIVE,
            'BOOKS AND MAGAZINES' AS BOOKS_AND_MAGAZINES,
            'SEAFOOD' AS SEAFOOD,
            'OFFICE SUPPLIES' AS OFFICE_SUPPLIES,
            'LAWN AND GARDEN' AS LAWN_AND_GARDEN,
            'SHEER HOSIERY' AS SHEER_HOSIERY,
            'WIRELESS' AS WIRELESS,
            'BEDDING' AS BEDDING,
            'BATH AND SHOWER' AS BATH_AND_SHOWER,
            'HORTICULTURE AND ACCESS' AS HORTICULTURE_AND_ACCESS,
            'HOME DECOR' AS HOME_DECOR,
            'TOYS' AS TOYS,
            'INFANT APPAREL' AS INFANT_APPAREL,
            'LADIES SOCKS' AS LADIES_SOCKS,
            'PLUS AND MATERNITY' AS PLUS_AND_MATERNITY,
            'ELECTRONICS' AS ELECTRONICS,
            'GIRLS WEAR, 4-6X  AND 7-14' AS GIRLS_WEAR__4_6X__AND_7_14,
            'BRAS & SHAPEWEAR' AS BRAS__SHAPEWEAR,
            'LIQUOR,WINE,BEER' AS LIQUOR_WINE_BEER,
            'SLEEPWEAR/FOUNDATIONS' AS SLEEPWEAR_FOUNDATIONS,
            'CAMERAS AND SUPPLIES' AS CAMERAS_AND_SUPPLIES,
            'SPORTING GOODS' AS SPORTING_GOODS,
            'PLAYERS AND ELECTRONICS' AS PLAYERS_AND_ELECTRONICS,
            'PHARMACY RX' AS PHARMACY_RX,
            'MENSWEAR' AS MENSWEAR,
            'OPTICAL - FRAMES' AS OPTICAL___FRAMES,
            'SWIMWEAR/OUTERWEAR' AS SWIMWEAR_OUTERWEAR,
            'OTHER DEPARTMENTS' AS OTHER_DEPARTMENTS,
            'MEDIA AND GAMING' AS MEDIA_AND_GAMING,
            'FURNITURE' AS FURNITURE,
            'OPTICAL - LENSES' AS OPTICAL___LENSES,
            'SEASONAL' AS SEASONAL,
            'LARGE HOUSEHOLD GOODS' AS LARGE_HOUSEHOLD_GOODS,
            '1-HR PHOTO' AS HR_PHOTO,
            'CONCEPT STORES' AS CONCEPT_STORES,
            'HEALTH AND BEAUTY AIDS' AS HEALTH_AND_BEAUTY_AIDS)
            )Tmp1
        ) WITH DATA;'''
    execute_sql(qry)

    # -- Pivot weekday
    qry = '''
        CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_scan_count AS (
        SEL o_order_id,
            quantity AS scan_count,
            TD_DAY_OF_WEEK(CAST (order_date AS DATE)) AS dayofweek,

            CASE
                WHEN dayofweek=1 THEN 'Sunday'
                WHEN dayofweek=2 THEN 'Monday'
                WHEN dayofweek=3 THEN 'Tuesday'
                WHEN dayofweek=4 THEN 'Wednesday'
                WHEN dayofweek=5 THEN 'Thursday'
                WHEN dayofweek=6 THEN 'Friday'
                WHEN dayofweek=7 THEN 'Saturday'
            END weekday
            FROM TPCXAI.uc08_order_lineitem_product
        ) WITH DATA;'''
    execute_sql(qry)

    # Query to create the order_lineitem_product_wd table by pivoting the weekday scan counts
    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_wd AS (
        SELECT o_order_id,

            CASE
                WHEN Sunday_sc > 0 THEN 1 ELSE 0
            END AS Sunday_sc,

            CASE
                WHEN Monday_sc > 0 THEN 1 ELSE 0
            END AS Monday_sc,

            CASE
                WHEN Tuesday_sc > 0 THEN 1 ELSE 0
            END AS Tuesday_sc,

            CASE
                WHEN Wednesday_sc > 0 THEN 1 ELSE 0
            END AS Wednesday_sc,

            CASE
                WHEN Thursday_sc > 0 THEN 1 ELSE 0
            END AS Thursday_sc,

            CASE
                WHEN Friday_sc > 0 THEN 1 ELSE 0
            END AS Friday_sc,

            CASE
                WHEN Saturday_sc > 0 THEN 1 ELSE 0
            END AS Saturday_sc
            FROM TPCXAI.uc08_order_lineitem_product_scan_count
            PIVOT (
            COUNT(scan_count) AS sc
            FOR weekday IN (
            'Sunday' AS Sunday,
            'Monday' AS Monday,
            'Tuesday' AS Tuesday,
            'Wednesday' AS Wednesday,
            'Thursday' AS Thursday,
            'Friday' AS Friday,
            'Saturday' AS Saturday)
            ) t
        ) WITH DATA;'''
    execute_sql(qry)

    # -- join aggregated and weekday table
    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_agg_wd2 AS (
        SELECT a.o_order_id,
            a.scan_count,
            a.scan_count_abs,
            a.trip_type,
            b.Sunday_sc,
            b.Monday_sc,
            b.Tuesday_sc,
            b.Wednesday_sc,
            b.Thursday_sc,
            b.Friday_sc,
            b.Saturday_sc
            FROM TPCXAI.uc08_order_lineitem_product_agg AS a JOIN
            TPCXAI.uc08_order_lineitem_product_wd b ON  a.o_order_id = b.o_order_id) WITH DATA;'''
    execute_sql(qry)

    # -- join the above table with dept table
    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_order_lineitem_product_agg_wd_dept AS (
        SELECT a.o_order_id,
            a.trip_type,
            a.scan_count,
            a.scan_count_abs,
            a.Sunday_sc,
            a.Monday_sc,
            a.Tuesday_sc,
            a.Wednesday_sc,
            a.Thursday_sc,
            a.Friday_sc,
            a.Saturday_sc,
            b.FINANCIAL_SERVICES_sc,
            b.SHOES_sc,
            b.PERSONAL_CARE_sc,
            b.PAINT_AND_ACCESSORIES_sc,
            b.DSD_GROCERY_sc,
            b.MEAT___FRESH__FROZEN_sc,
            b.DAIRY_sc,
            b.PETS_AND_SUPPLIES_sc,
            b.HOUSEHOLD_CHEMICALS_SUPP_sc,
            b.IMPULSE_MERCHANDISE_sc,
            b.PRODUCE_sc,
            b.CANDY__TOBACCO__COOKIES_sc,
            b.GROCERY_DRY_GOODS_sc,
            b.BOYS_WEAR_sc,
            b.FABRICS_AND_CRAFTS_sc,
            b.JEWELRY_AND_SUNGLASSES_sc,
            b.MENS_WEAR_sc,
            b.ACCESSORIES_sc,
            b.HOME_MANAGEMENT_sc,
            b.FROZEN_FOODS_sc,
            b.SERVICE_DELI_sc,
            b.INFANT_CONSUMABLE_HARDLINES_sc,
            b.PRE_PACKED_DELI_sc,
            b.COOK_AND_DINE_sc,
            b.PHARMACY_OTC_sc,
            b.LADIESWEAR_sc,
            b.COMM_BREAD_sc,
            b.BAKERY_sc,
            b.HOUSEHOLD_PAPER_GOODS_sc,
            b.CELEBRATION_sc,
            b.HARDWARE_sc,
            b.BEAUTY_sc,
            b.AUTOMOTIVE_sc,
            b.BOOKS_AND_MAGAZINES_sc,
            b.SEAFOOD_sc,
            b.OFFICE_SUPPLIES_sc,
            b.LAWN_AND_GARDEN_sc,
            b.SHEER_HOSIERY_sc,
            b.WIRELESS_sc,
            b.BEDDING_sc,
            b.BATH_AND_SHOWER_sc,
            b.HORTICULTURE_AND_ACCESS_sc,
            b.HOME_DECOR_sc,
            b.TOYS_sc,
            b.INFANT_APPAREL_sc,
            b.LADIES_SOCKS_sc,
            b.PLUS_AND_MATERNITY_sc,
            b.ELECTRONICS_sc,
            b.GIRLS_WEAR__4_6X__AND_7_14_sc,
            b.BRAS__SHAPEWEAR_sc,
            b.LIQUOR_WINE_BEER_sc,
            b.SLEEPWEAR_FOUNDATIONS_sc,
            b.CAMERAS_AND_SUPPLIES_sc,
            b.SPORTING_GOODS_sc,
            b.PLAYERS_AND_ELECTRONICS_sc,
            b.PHARMACY_RX_sc,
            b.MENSWEAR_sc,
            b.OPTICAL___FRAMES_sc,
            b.SWIMWEAR_OUTERWEAR_sc,
            b.OTHER_DEPARTMENTS_sc,
            b.MEDIA_AND_GAMING_sc,
            b.FURNITURE_sc,
            b.OPTICAL___LENSES_sc,
            b.SEASONAL_sc,
            b.LARGE_HOUSEHOLD_GOODS_sc,
            b.HR_PHOTO_sc,
            b.CONCEPT_STORES_sc,
            b.HEALTH_AND_BEAUTY_AIDS_sc
            FROM TPCXAI.uc08_order_lineitem_product_agg_wd2 AS a JOIN
            TPCXAI.uc08_order_lineitem_product_pivot_dept b ON a.o_order_id = b.o_order_id) WITH DATA;'''
    execute_sql(qry)

    # -- prepare final pre-processed table
    qry = '''
    CREATE MULTISET TABLE TPCXAI.uc08_preprocessed AS (
        SELECT
            o_order_id,
            scan_count,
            scan_count_abs,
            trip_type,
            Sunday_sc AS Sunday,
            Monday_sc AS Monday,
            Tuesday_sc AS Tuesday,
            Wednesday_sc AS Wednesday,
            Thursday_sc Thursday,
            Friday_sc AS Friday,
            Saturday_sc AS Saturday,
            COALESCE(FINANCIAL_SERVICES_sc,0) AS FINANCIAL_SERVICES,
            COALESCE(SHOES_sc,0) AS SHOES,
            COALESCE(PERSONAL_CARE_sc,0) AS PERSONAL_CARE,
            COALESCE(PAINT_AND_ACCESSORIES_sc,0) AS PAINT_AND_ACCESSORIES,
            COALESCE(DSD_GROCERY_sc,0) AS DSD_GROCERY,
            COALESCE(MEAT___FRESH__FROZEN_sc,0) AS MEAT___FRESH__FROZEN,
            COALESCE(DAIRY_sc,0) AS DAIRY,
            COALESCE(PETS_AND_SUPPLIES_sc,0) AS PETS_AND_SUPPLIES,
            COALESCE(HOUSEHOLD_CHEMICALS_SUPP_sc,0) AS HOUSEHOLD_CHEMICALS_SUPP,
            COALESCE(IMPULSE_MERCHANDISE_sc,0) AS IMPULSE_MERCHANDISE,
            COALESCE(PRODUCE_sc,0) AS PRODUCE,
            COALESCE(CANDY__TOBACCO__COOKIES_sc,0) AS CANDY__TOBACCO__COOKIES,
            COALESCE(GROCERY_DRY_GOODS_sc,0) AS GROCERY_DRY_GOODS,
            COALESCE(BOYS_WEAR_sc,0) AS BOYS_WEAR,
            COALESCE(FABRICS_AND_CRAFTS_sc,0) AS FABRICS_AND_CRAFTS,
            COALESCE(JEWELRY_AND_SUNGLASSES_sc,0) AS JEWELRY_AND_SUNGLASSES,
            COALESCE(MENS_WEAR_sc,0) AS MENS_WEAR,
            COALESCE(ACCESSORIES_sc,0) AS ACCESSORIES,
            COALESCE(HOME_MANAGEMENT_sc,0) AS HOME_MANAGEMENT,
            COALESCE(FROZEN_FOODS_sc,0) AS FROZEN_FOODS,
            COALESCE(SERVICE_DELI_sc,0) AS SERVICE_DELI,
            COALESCE(INFANT_CONSUMABLE_HARDLINES_sc,0) AS INFANT_CONSUMABLE_HARDLINES,
            COALESCE(PRE_PACKED_DELI_sc,0) AS PRE_PACKED_DELI,
            COALESCE(COOK_AND_DINE_sc,0) AS COOK_AND_DINE,
            COALESCE(PHARMACY_OTC_sc,0) AS PHARMACY_OTC,
            COALESCE(LADIESWEAR_sc,0) AS LADIESWEAR,
            COALESCE(COMM_BREAD_sc,0) AS COMM_BREAD,
            COALESCE(BAKERY_sc,0) AS BAKERY,
            COALESCE(HOUSEHOLD_PAPER_GOODS_sc,0) AS HOUSEHOLD_PAPER_GOODS,
            COALESCE(CELEBRATION_sc,0) AS CELEBRATION,
            COALESCE(HARDWARE_sc,0) AS HARDWARE,
            COALESCE(BEAUTY_sc,0) AS BEAUTY,
            COALESCE(AUTOMOTIVE_sc,0) AS AUTOMOTIVE,
            COALESCE(BOOKS_AND_MAGAZINES_sc,0) AS BOOKS_AND_MAGAZINES,
            COALESCE(SEAFOOD_sc,0) AS SEAFOOD,
            COALESCE(OFFICE_SUPPLIES_sc,0) AS OFFICE_SUPPLIES,
            COALESCE(LAWN_AND_GARDEN_sc,0) AS LAWN_AND_GARDEN,
            COALESCE(SHEER_HOSIERY_sc,0) AS SHEER_HOSIERY,
            COALESCE(WIRELESS_sc,0) AS WIRELESS,
            COALESCE(BEDDING_sc,0) AS BEDDING,
            COALESCE(BATH_AND_SHOWER_sc,0) AS BATH_AND_SHOWER,
            COALESCE(HORTICULTURE_AND_ACCESS_sc,0) AS HORTICULTURE_AND_ACCESS,
            COALESCE(HOME_DECOR_sc,0) AS HOME_DECOR,
            COALESCE(TOYS_sc,0) AS TOYS,
            COALESCE(INFANT_APPAREL_sc,0) AS INFANT_APPAREL,
            COALESCE(LADIES_SOCKS_sc,0) AS LADIES_SOCKS,
            COALESCE(PLUS_AND_MATERNITY_sc,0) AS PLUS_AND_MATERNITY,
            COALESCE(ELECTRONICS_sc,0) AS ELECTRONICS,
            COALESCE(GIRLS_WEAR__4_6X__AND_7_14_sc,0) AS GIRLS_WEAR__4_6X__AND_7_14,
            COALESCE(BRAS__SHAPEWEAR_sc,0) AS BRAS__SHAPEWEAR,
            COALESCE(LIQUOR_WINE_BEER_sc,0) AS LIQUOR_WINE_BEER,
            COALESCE(SLEEPWEAR_FOUNDATIONS_sc,0) AS SLEEPWEAR_FOUNDATIONS,
            COALESCE(CAMERAS_AND_SUPPLIES_sc,0) AS CAMERAS_AND_SUPPLIES,
            COALESCE(SPORTING_GOODS_sc,0) AS SPORTING_GOODS,
            COALESCE(PLAYERS_AND_ELECTRONICS_sc,0) AS PLAYERS_AND_ELECTRONICS,
            COALESCE(PHARMACY_RX_sc,0) AS PHARMACY_RX,
            COALESCE(MENSWEAR_sc,0) AS MENSWEAR,
            COALESCE(OPTICAL___FRAMES_sc,0) AS OPTICAL___FRAMES,
            COALESCE(SWIMWEAR_OUTERWEAR_sc,0) AS SWIMWEAR_OUTERWEAR,
            COALESCE(OTHER_DEPARTMENTS_sc,0) AS OTHER_DEPARTMENTS,
            COALESCE(MEDIA_AND_GAMING_sc,0) AS MEDIA_AND_GAMING,
            COALESCE(FURNITURE_sc,0) AS FURNITURE,
            COALESCE(OPTICAL___LENSES_sc,0) AS OPTICAL___LENSES,
            COALESCE(SEASONAL_sc,0) AS SEASONAL,
            COALESCE(LARGE_HOUSEHOLD_GOODS_sc,0) AS LARGE_HOUSEHOLD_GOODS,
            COALESCE(HR_PHOTO_sc,0) AS HR_PHOTO,
            COALESCE(CONCEPT_STORES_sc,0) AS CONCEPT_STORES,
            COALESCE(HEALTH_AND_BEAUTY_AIDS_sc,0) AS HEALTH_AND_BEAUTY_AIDS
            FROM TPCXAI.uc08_order_lineitem_product_agg_wd_dept where trip_type <> 14)
        WITH DATA;
    '''

    return execute_sql(qry) 


<h3><b>Define a function to train the data on an XGBoost model</b></h3>
<p>Use the in database analytic function: TD_XGBoost() to train an XGBoost model on the preprocessed data.</p>

In [13]:
# Function to train the XGBoost model using the preprocessed data
def train(model_name):

    # Drop the table if it already exists to avoid conflicts during creation
    tables = [model_name, f"{model_name}_meta"]
    for t in tables:
        try:
            print("deleting table: ", t)
            #eng.execute(f"DROP TABLE {t}")
            execute_sql(f"DROP TABLE TPCXAI.{t}")
        except:
            print("Error deleting table: ", t)
            pass

    # Query to create the XGBoost model using the preprocessed data
    qry = f'''
        CREATE MULTISET TABLE TPCXAI.{model_name} AS (
        SELECT *
            FROM TD_XGBoost(
            ON TPCXAI.uc08_preprocessed PARTITION BY ANY
            OUT TABLE MetaInformationTable(TPCXAI.{model_name}_meta)
            USING
            ResponseColumn('trip_type')
            InputColumns('scan_count','scan_count_abs','Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','FINANCIAL_SERVICES','SHOES','PERSONAL_CARE','PAINT_AND_ACCESSORIES','DSD_GROCERY','MEAT___FRESH__FROZEN','DAIRY','PETS_AND_SUPPLIES','HOUSEHOLD_CHEMICALS_SUPP','IMPULSE_MERCHANDISE','PRODUCE','CANDY__TOBACCO__COOKIES','GROCERY_DRY_GOODS','BOYS_WEAR','FABRICS_AND_CRAFTS','JEWELRY_AND_SUNGLASSES','MENS_WEAR','ACCESSORIES','HOME_MANAGEMENT','FROZEN_FOODS','SERVICE_DELI','INFANT_CONSUMABLE_HARDLINES','PRE_PACKED_DELI','COOK_AND_DINE','PHARMACY_OTC','LADIESWEAR','COMM_BREAD','BAKERY','HOUSEHOLD_PAPER_GOODS','CELEBRATION','HARDWARE','BEAUTY','AUTOMOTIVE','BOOKS_AND_MAGAZINES','SEAFOOD','OFFICE_SUPPLIES','LAWN_AND_GARDEN','SHEER_HOSIERY','WIRELESS','BEDDING','BATH_AND_SHOWER','HORTICULTURE_AND_ACCESS','HOME_DECOR','TOYS','INFANT_APPAREL','LADIES_SOCKS','PLUS_AND_MATERNITY','ELECTRONICS','GIRLS_WEAR__4_6X__AND_7_14','BRAS__SHAPEWEAR','LIQUOR_WINE_BEER','SLEEPWEAR_FOUNDATIONS','CAMERAS_AND_SUPPLIES','SPORTING_GOODS','PLAYERS_AND_ELECTRONICS','PHARMACY_RX','MENSWEAR','OPTICAL___FRAMES','SWIMWEAR_OUTERWEAR','OTHER_DEPARTMENTS','MEDIA_AND_GAMING','FURNITURE','OPTICAL___LENSES','SEASONAL','LARGE_HOUSEHOLD_GOODS','HR_PHOTO','CONCEPT_STORES','HEALTH_AND_BEAUTY_AIDS')
            MaxDepth(3)
            MinNodeSize(1)
            NumBoostedTrees(75)
            ModelType('classification')
            Seed(123)
            ShrinkageFactor(0.1)
            IterNum(10)
            ColumnSampling(1.0)
            ) AS dt) WITH DATA;
    '''

    try:
        #eng.execute(qry)
        execute_sql(qry)
        print("Model trained and table created..")
        return "Model trained successfully.."
    except:
        print("Issue in model training..")
        return "Sorry, there is some issue in model training"



<h3><b>Define a function to serve the data to the model</b></h3>
<p>Use the in database function: TD_XGBoostPredict to run the trained XGBoost model.</p>

In [19]:
# Function to serve the XGBoost model using the preprocessed data and generate predictions in a new table
def serve(model_name):
    #print('serve called...')
    # drop the table first
    try:
        #eng.execute("DROP TABLE uc08_pysql_model_predict")
        execute_sql("DROP TABLE TPCXAI.uc08_pysql_model_predict")
        print("Table dropped: uc08_pysql_model_predict")
    except:
        print("No table to drop: uc08_pysql_model_predict")
        pass

    # Query to create the prediction table using the XGBoost model and preprocessed data
    qry = f'''
    CREATE TABLE TPCXAI.uc08_pysql_model_predict AS (
    SELECT *
        FROM TD_XGBoostPredict(
        ON TPCXAI.uc08_preprocessed AS inputtable PARTITION BY ANY
        ON TPCXAI.{model_name} AS modeltable DIMENSION
        ORDER BY task_index,
            tree_num,
            iter,
            class_num,
            tree_order
        USING
        IdColumn('o_order_id')
        NumParallelTrees(75)
        NumBoostRounds(7)
        ModelType('classification')
        Accumulate('trip_type')
        ) AS dt) WITH DATA;
    '''
    #print('Query saved, executing now...')
    
    try:
        #eng.execute(qry)
        execute_sql(qry)
        print("Fetching prediction results...")
    except:
        print("Sorry, there is some issue in model serving")


<h3><b>Define a function to show accuracy metrics</b></h3>

In [15]:
# Function to show accuracy of the model by comparing actual and predicted trip types
def show_accuracy():
    # Get the number of unique labels
    num_labels = DataFrame.from_query("SELECT COUNT(DISTINCT trip_type) AS cnt FROM TPCXAI.uc08_preprocessed").get_values(num_rows=1)[0][0]
    #print(f"Number of unique labels (trip_type): {num_labels}")

    # Accuracy calculation query
    accuracy_qry = f'''
        SELECT * FROM TD_ClassificationEvaluator (
            ON TPCXAI.uc08_pysql_model_predict AS InputTable
            OUT TABLE OutputTable (TPCXAI.uc08_accuracy)
            USING
            ObservationColumn('trip_type')
            PredictionColumn('Prediction')
            NumLabels({num_labels})
        ) AS dt;
    '''
    #print('show_accuracy called...')
    try:
        execute_sql("DROP TABLE TPCXAI.uc08_accuracy")
        print("Table dropped: uc08_accuracy")
    except:
        pass

    execute_sql(accuracy_qry)
    print("Accuracy table created..")

<h3><b>Define a main function to pre process, train, and serve the model</b></h3>
<p>This function calls all the above functions to:</p>
<ol>
<li>Connect to the database</li>
<li>Preprocess the data</li>
<li>Train the XGBoost model on the preprocessed data</li>
<li>Serve the XGBoost model</li>
</ol>

In [20]:
# Run this section NOT the below function
# Main function to execute the entire workflow from data preprocessing to model training and serving
def main():
    start_main = timeit.default_timer()

    # Connect to the database
    start_connect = timeit.default_timer()
    connect_to_db()
    end_connect = timeit.default_timer()
    print(f"Database connection established in {end_connect - start_connect} seconds.")

    # Preprocess the data
    start_preprocess = timeit.default_timer()
    pre_process()
    end_preprocess = timeit.default_timer()
    print(f"Data preprocessing completed in {end_preprocess - start_preprocess} seconds.")

    # Create the XGBoost model
    start_train = timeit.default_timer()
    model_name = "classify_trips"
    train(model_name)
    end_train = timeit.default_timer()
    print(f"Model training completed in {end_train - start_train} seconds.")

    # Serve the model
    start_serve = timeit.default_timer()
    serve(model_name)
    end_serve = timeit.default_timer()
    print(f"Model serving completed in {end_serve - start_serve} seconds.")

    # Show accuracy
    start_accuracy = timeit.default_timer()
    show_accuracy()
    end_accuracy = timeit.default_timer()
    print(f"Accuracy evaluation completed in {end_accuracy - start_accuracy} seconds.")

    end_main = timeit.default_timer()
    print(f"Main function executed in {end_main - start_main} seconds.")


<h4>Run the main function</h4>

In [21]:
# Run the main function
main()

c:\Users\jv255027\AppData\Local\miniforge3\envs\UseCase8env\Lib\site-packages\teradatasqlalchemy\telemetry\queryband.py:382: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)


Database connection established in 7.161364899948239 seconds.
Data preprocessing completed in 54.2109052001033 seconds.
deleting table:  classify_trips
deleting table:  classify_trips_meta
Model trained and table created..
Model training completed in 8.06584019982256 seconds.
Table dropped: uc08_pysql_model_predict
Fetching prediction results...
Model serving completed in 267.78859689994715 seconds.
Table dropped: uc08_accuracy
Accuracy table created..
Accuracy evaluation completed in 3.429693000158295 seconds.
Main function executed in 340.6568744999822 seconds.


In [22]:
# View accuracy
accuracy_query = DataFrame.from_query("SELECT * FROM TPCXAI.uc08_accuracy")
accuracy_query

SeqNum,Metric,MetricValue
3,Micro-Recall,0.6595289013339077
5,Macro-Precision,0.7047622780825544
6,Macro-Recall,0.4825090170270561
7,Macro-F1,0.5131277899327177
9,Weighted-Recall,0.6595289013339077
10,Weighted-F1,0.611051826867912
8,Weighted-Precision,0.6601501054461394
4,Micro-F1,0.6595289013339077
2,Micro-Precision,0.6595289013339077
1,Accuracy,0.6595289013339077
